# Traffic Demand Prediction

We predict the `demand` for each road location and time. The score is `100 x R-squared`, so getting the busy roads right matters most.

The model is XGBoost. Three things made it work well:
1. We use both days of training data together.
2. For every location we add its average past demand as a feature.
3. We tell the model to care more about the busy roads, and we train it seven times and average the results.

This scores about 91.2 on the leaderboard.

In [ ]:
import numpy as np
import pandas as pd
import pygeohash as pgh
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

## 1. Load the data

The training file has two days of records. We keep both. We also save the test `Index` column so the submission rows line up correctly.

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/habisrizvipc/hackethon/train.csv')
test  = pd.read_csv('/kaggle/input/datasets/habisrizvipc/hackethon/test.csv')

test_index = test['Index'].copy()

print('Train shape:', train.shape)
print('Test shape :', test.shape)
train.head()

## 2. Location averages

The strongest signal is the location itself. A highway is always busy, a quiet street is not. So for every location we work out its average demand (and how much it varies) from the training data, and we use that as a feature.

We work this out once, from the training data, and reuse it for the test data. New locations that never appear in training just get the overall average.

In [ ]:
geohash_stats = train.groupby('geohash')['demand'].agg(
    geohash_mean='mean', geohash_std='std'
).reset_index()
geohash_stats['geohash_std'] = geohash_stats['geohash_std'].fillna(0)

global_mean = geohash_stats['geohash_mean'].mean()

## 3. Build the features

For each row we add:
- **Time of day** in minutes, plus sine and cosine of the hour so that 11pm sits next to midnight instead of far away.
- **A rush-hour flag** for the usual busy hours.
- **Latitude and longitude** decoded from the location code.
- **The location average and spread** from the step above.
- **Lanes on a highway**, which picks out the big roads.
- The road type, vehicle, landmark and weather columns turned into simple yes/no columns.

Missing values are filled with sensible defaults.

In [ ]:
def build_features(df, geohash_stats, global_mean):
    df = df.copy()

    # Split the timestamp into hour and minute
    df[['hour', 'minute']] = df['timestamp'].str.split(':', expand=True).astype(int)
    df['time_of_day'] = df['hour'] * 60 + df['minute']

    # Sine/cosine so the clock wraps around cleanly
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    # Rush hour flag
    df['is_peak'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)

    # Location code -> latitude and longitude
    df['lat'] = df['geohash'].apply(lambda x: pgh.decode(x)[0])
    df['lon'] = df['geohash'].apply(lambda x: pgh.decode(x)[1])

    # Attach the location averages
    df = df.merge(geohash_stats, on='geohash', how='left')
    df['geohash_mean'] = df['geohash_mean'].fillna(global_mean)
    df['geohash_std']  = df['geohash_std'].fillna(0)

    # Lanes on a highway
    df['lanes_x_highway'] = df['NumberofLanes'] * (df['RoadType'] == 'Highway').astype(int)

    # Fill gaps
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
    df['Weather']     = df['Weather'].fillna('Sunny')
    df['RoadType']    = df['RoadType'].fillna('Residential')

    # Turn the text columns into yes/no columns
    df = pd.get_dummies(df, columns=['RoadType', 'LargeVehicles', 'Landmarks', 'Weather'],
                        drop_first=True)

    df = df.drop(['geohash', 'timestamp'], axis=1)
    return df


train_fe = build_features(train, geohash_stats, global_mean)
test_fe  = build_features(test,  geohash_stats, global_mean)

print('Train after features:', train_fe.shape)
print('Test  after features:', test_fe.shape)

## 4. Line up the columns

The yes/no columns can differ slightly between train and test, so we force the test set to use exactly the same columns.

In [ ]:
X = train_fe.drop(['demand', 'Index'], axis=1)
y = train_fe['demand']

test_fe = test_fe.drop('Index', axis=1)
test_fe = test_fe.reindex(columns=X.columns, fill_value=0)

print('Columns match:', list(X.columns) == list(test_fe.columns))

## 5. Train the model

Two choices here are what lift the score:

**Give busy roads more weight.** Most roads are quiet, so a plain model plays it safe and guesses low everywhere, which leaves the busy highways too low. We give each row a weight of `1 + 5 x demand`. A busy highway then gets a weight of about 4 and a quiet road about 1.3, so the busy road counts roughly three times as much. The model learns to push the busy roads up, and those are the rows the score cares about.

*How we landed on 5:* we tried a range of strengths on a slice of data the model had not trained on, and checked which scored best. Weighting by 3, 5 or 8 all came out very close. More weight kept helping by a hair, but pushing it too far (10 and above) started to hurt the quiet roads. So 5 is a safe middle: strong enough to fix the busy-road problem, not so strong that it chases noise or ignores everything else.

**Train seven times and average.** Each run uses a different random split and a different random seed, so each one is a little different. Averaging the seven runs cancels out the random noise and gives a steadier prediction.

We keep the tree depth at 6. Deeper trees looked tempting in testing but did not help on the real leaderboard, so we left it alone.

In [ ]:
N_RUNS = 7
WEIGHT = 5.0   # busy-road weight: each row counts as 1 + WEIGHT * demand

test_preds = np.zeros(len(test_fe))

for seed in range(N_RUNS):
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    weights = 1 + WEIGHT * y_train

    model = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_alpha=0.1,
        reg_lambda=1.0,
        early_stopping_rounds=50,
        tree_method='hist',
        device='cuda',          # change to 'cpu' if there is no GPU
        random_state=seed
    )
    model.fit(
        X_train, y_train,
        sample_weight=weights,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    preds = np.clip(model.predict(test_fe), 0, 1)
    test_preds += preds
    print(f'Run {seed + 1} of {N_RUNS} done')

test_preds /= N_RUNS

## 6. Quick check on held-out data

A simple check on a 20% slice we did not train the last model on, just to see the score is sensible.

In [ ]:
valid_preds = np.clip(model.predict(X_valid), 0, 1)
r2 = r2_score(y_valid, valid_preds)
print(f'R-squared on the held-out slice: {r2:.4f}')
print(f'Score estimate: {max(0, 100 * r2):.2f}')

## 7. Save the submission

Predictions are kept between 0 and 1, and the rows use the saved test `Index`.

In [ ]:
submission = pd.DataFrame({
    'Index': test_index,
    'demand': test_preds
})

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
print(submission.head())